In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_community.tools.tavily_search import TavilySearchResults

tools = [TavilySearchResults(max_results = 3)]

C:\Users\USER\AppData\Local\Temp\ipykernel_9272\2432580975.py:3: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tools = [TavilySearchResults(max_results = 3)]


In [3]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

llm = ChatOpenAI(model = 'gpt-4o')
agent_executor = create_agent(llm, tools)

In [6]:
resp = agent_executor.invoke({'messages':['user','LangChain의 개요를 설명해줘']})

In [7]:
resp

{'messages': [HumanMessage(content='user', additional_kwargs={}, response_metadata={}, id='1112eafe-11b2-4185-b1b2-22e7748be360'),
  HumanMessage(content='LangChain의 개요를 설명해줘', additional_kwargs={}, response_metadata={}, id='8572bdcb-34e9-4111-a071-a01e6b5787df'),
  AIMessage(content='LangChain은 자연어 처리(NLP) 및 인공지능(AI) 응용 프로그램을 개발하는 데 유용한 Python 기반 프레임워크입니다. 이 툴킷은 특히 대화형 AI, 자동화된 응답 시스템, 텍스트 생성 및 분석, 정보 검색, 챗봇 구축 등에 특화되어 있습니다. LangChain의 주요 특징과 기능은 다음과 같습니다:\n\n1. **모듈성**: LangChain은 다양한 구성 요소를 모듈화하여 사용자들이 필요한 기능을 손쉽게 사용할 수 있도록 설계되었습니다. 이러한 구성 요소는 언어 모델, 데이터베이스 연결, 응답 생성 등의 기능을 포함합니다.\n\n2. **유연성**: 다양한 AI 모델과 데이터 소스를 통합할 수 있도록 설계되어, 사용자의 특정 요구에 맞춘 커스터마이징이 가능합니다. 예를 들어, OpenAI의 GPT 모델과 같은 언어 모델을 쉽게 연동할 수 있습니다.\n\n3. **확장성**: 대규모 AI 응용 프로그램을 구축할 수 있도록 설계되어, 복잡한 비즈니스 요구에도 대응할 수 있습니다. 많은 데이터와 사용자 요청을 효율적으로 처리할 수 있는 강력한 프레임워크를 제공합니다.\n\n4. **다양한 활용 사례**: LangChain은 정보 검색, 데이터 요약, 자동 보고서 생성, 고객 지원 등을 포함한 다양한 영역에 적용될 수 있습니다.\n\n5. **사용자 친화적인 API**: 개발자가 쉽게 접근하고 사용할 수 있도록 직관적인 API를 제공하여, 복잡한 언

In [8]:
from langchain_community.document_loaders import GitLoader


def file_filter(file_path: str) -> bool:
    return file_path.endswith(".md")


loader = GitLoader(
    clone_url="https://github.com/langchain-ai/langchain",
    repo_path="./langchain",
    branch="master",
    file_filter=file_filter,
)

documents = loader.load()
print(len(documents))

36


In [9]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)

documents = text_splitter.split_documents(documents)
print(len(documents))

Created a chunk of size 1379, which is longer than the specified 1000
Created a chunk of size 1455, which is longer than the specified 1000
Created a chunk of size 1299, which is longer than the specified 1000


96


In [10]:
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_community.embeddings import HuggingFaceEmbeddings

In [11]:
embeddings_model = HuggingFaceEmbeddings(
    model_name='jhgan/ko-sbert-nli',
    model_kwargs={'device':'cpu'},
    encode_kwargs={'normalize_embeddings':True},
)


vectorstore = FAISS.from_documents(documents,
                                   embedding = embeddings_model,
                                   distance_strategy = DistanceStrategy.COSINE
                                  )

C:\Users\USER\AppData\Local\Temp\ipykernel_9272\707567803.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings_model = HuggingFaceEmbeddings(


In [13]:
query = 'Langchain의 개요를 알려줘.'

In [14]:
retriever = vectorstore.as_retriever(search_kwargs = {'k':1})
docs = retriever.invoke(query)

In [15]:
retriever = vectorstore.as_retriever(search_kwargs = {'k':1})
docs = retriever.invoke(query)
print(len(docs))
print(docs[0])

1
page_content='```txt
langchain/
├── libs/
│   ├── core/             # `langchain-core` primitives and base abstractions
│   ├── langchain/        # `langchain-classic` (legacy, no new features)
│   ├── langchain_v1/     # Actively maintained `langchain` package
│   ├── partners/         # Third-party integrations
│   │   ├── openai/       # OpenAI models and embeddings
│   │   ├── anthropic/    # Anthropic (Claude) integration
│   │   ├── ollama/       # Local model support
│   │   └── ... (other integrations maintained by the LangChain team)
│   ├── text-splitters/   # Document chunking utilities
│   ├── standard-tests/   # Shared test suite for integrations
│   ├── model-profiles/   # Model configuration profiles
│   └── cli/              # Command-line interface tools
├── .github/              # CI/CD workflows and templates
├── .vscode/              # VSCode IDE standard settings and recommended extensions
└── README.md             # Information about LangChain
```' metadata={'so

In [16]:
retriever = vectorstore.as_retriever(search_type = 'mmr', search_kwargs = {'k':5, 'fetch_k':50}) # 후보 수 50개 중 상위 5개
docs = retriever.invoke(query)
print(len(docs))
print(docs[0])

5
page_content='```txt
langchain/
├── libs/
│   ├── core/             # `langchain-core` primitives and base abstractions
│   ├── langchain/        # `langchain-classic` (legacy, no new features)
│   ├── langchain_v1/     # Actively maintained `langchain` package
│   ├── partners/         # Third-party integrations
│   │   ├── openai/       # OpenAI models and embeddings
│   │   ├── anthropic/    # Anthropic (Claude) integration
│   │   ├── ollama/       # Local model support
│   │   └── ... (other integrations maintained by the LangChain team)
│   ├── text-splitters/   # Document chunking utilities
│   ├── standard-tests/   # Shared test suite for integrations
│   ├── model-profiles/   # Model configuration profiles
│   └── cli/              # Command-line interface tools
├── .github/              # CI/CD workflows and templates
├── .vscode/              # VSCode IDE standard settings and recommended extensions
└── README.md             # Information about LangChain
```' metadata={'so

mmr 계수  
lambda가 작을 수록 중복을 줄이고 다양하게 추출

In [17]:
retriever = vectorstore.as_retriever(search_type = 'mmr', search_kwargs = {'k':5, 'lambda_mult':0.15}) # 후보 수 50개 중 상위 5개
docs = retriever.invoke(query)
print(len(docs))
print(docs[0])

5
page_content='```txt
langchain/
├── libs/
│   ├── core/             # `langchain-core` primitives and base abstractions
│   ├── langchain/        # `langchain-classic` (legacy, no new features)
│   ├── langchain_v1/     # Actively maintained `langchain` package
│   ├── partners/         # Third-party integrations
│   │   ├── openai/       # OpenAI models and embeddings
│   │   ├── anthropic/    # Anthropic (Claude) integration
│   │   ├── ollama/       # Local model support
│   │   └── ... (other integrations maintained by the LangChain team)
│   ├── text-splitters/   # Document chunking utilities
│   ├── standard-tests/   # Shared test suite for integrations
│   ├── model-profiles/   # Model configuration profiles
│   └── cli/              # Command-line interface tools
├── .github/              # CI/CD workflows and templates
├── .vscode/              # VSCode IDE standard settings and recommended extensions
└── README.md             # Information about LangChain
```' metadata={'so

In [ ]:
retriever = vectorstore.as_retriever(search_type = 'similarity_score_threshold', search_kwargs = {'score_threshold':0.1}) # -1 ~ 1 사이 값
print(len(docs))
print(docs[0])

4
page_content='```txt
langchain/
├── libs/
│   ├── core/             # `langchain-core` primitives and base abstractions
│   ├── langchain/        # `langchain-classic` (legacy, no new features)
│   ├── langchain_v1/     # Actively maintained `langchain` package
│   ├── partners/         # Third-party integrations
│   │   ├── openai/       # OpenAI models and embeddings
│   │   ├── anthropic/    # Anthropic (Claude) integration
│   │   ├── ollama/       # Local model support
│   │   └── ... (other integrations maintained by the LangChain team)
│   ├── text-splitters/   # Document chunking utilities
│   ├── standard-tests/   # Shared test suite for integrations
│   ├── model-profiles/   # Model configuration profiles
│   └── cli/              # Command-line interface tools
├── .github/              # CI/CD workflows and templates
├── .vscode/              # VSCode IDE standard settings and recommended extensions
└── README.md             # Information about LangChain
```' metadata={'so

In [19]:
# 유사도 값을 확인하고 싶습니다ㅏ.
result = vectorstore.similarity_search_with_score(query=query, k = 4)

In [20]:
for doc, score in result:
    print(score)

0.7702615
0.7702615
0.7834461
0.8049631


In [21]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


# Retrieval
retriever = vectorstore.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 5, 'lambda_mult': 0.15}
)

docs = retriever.invoke(query)

# Prompt
template = '''Answer the question based only on the following context:
{context}

Question: {question}
'''

prompt = ChatPromptTemplate.from_template(template)

# Model
llm = ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0,
    max_tokens=500,
)


def format_docs(docs):
    return '\n\n'.join([d.page_content for d in docs])

# Chain
chain = prompt | llm | StrOutputParser()

# Run
response = chain.invoke({'context': (format_docs(docs)), 'question':query})
response

'LangChain은 LLM(대형 언어 모델)을 활용한 애플리케이션을 개발하는 데 도움을 주는 도구입니다. 표준 인터페이스를 통해 모델, 임베딩, 벡터 저장소 등을 관리할 수 있도록 설계되었습니다. LangChain은 다양한 통합 기능을 제공하며, 개발자들이 LLM을 쉽게 사용할 수 있도록 지원합니다. 이 프로젝트는 오픈 소스이며, 새로운 기능, 인프라 개선, 문서 개선 등 다양한 기여를 환영합니다.'

### ChromaDB 사용

In [23]:
from langchain_community.document_loaders import GitLoader


def file_filter(file_path: str) -> bool:
    return file_path.endswith(".md")


loader = GitLoader(
    clone_url="https://github.com/langchain-ai/langchain",
    repo_path="./langchain",
    branch="master",
    file_filter=file_filter,
)

documents = loader.load()
print(len(documents))

from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)

docs = text_splitter.split_documents(documents)

from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

from langchain_chroma import Chroma

db = Chroma.from_documents(docs, embeddings)

retriever = db.as_retriever()



# 가장 유사도가 높은 문장을 하나만 추출
retriever = db.as_retriever(search_kwargs={'k': 1})

docs = retriever.invoke(query)
print(len(docs))
print(docs[0])


# MMR - 다양성 고려 (lambda_mult = 0.5)
retriever = db.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 5, 'fetch_k': 50}
)

docs = retriever.invoke(query)
print(len(docs))
docs[0]


from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


# Retrieval
retriever = db.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 5, 'lambda_mult': 0.15}
)

docs = retriever.invoke(query)

# Prompt
template = '''Answer the question based only on the following context:
{context}

Question: {question}
'''

prompt = ChatPromptTemplate.from_template(template)

# Model
llm = ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0,
    max_tokens=500,
)


def format_docs(docs):
    return '\n\n'.join([d.page_content for d in docs])

# Chain
chain = prompt | llm | StrOutputParser()

# Run
response = chain.invoke({'context': (format_docs(docs)), 'question':query})
response

36
1
page_content='LangChain is a framework for building agents and LLM-powered applications. It helps you chain together interoperable components and third-party integrations to simplify AI application development – all while future-proofing decisions as the underlying technology evolves.

```bash
pip install langchain
```

If you're looking for more advanced customization or agent orchestration, check out [LangGraph](https://docs.langchain.com/oss/python/langgraph/overview), our framework for building controllable agent workflows.

---

**Documentation**:

- [docs.langchain.com](https://docs.langchain.com/oss/python/langchain/overview) – Comprehensive documentation, including conceptual overviews and guides
- [reference.langchain.com/python](https://reference.langchain.com/python) – API reference docs for LangChain packages

**Discussions**: Visit the [LangChain Forum](https://forum.langchain.com) to connect with the community and share all of your technical questions, ideas, and fee

'LangChain은 에이전트 및 LLM 기반 애플리케이션을 구축하기 위한 프레임워크입니다. 이 프레임워크는 상호 운용 가능한 구성 요소와 제3자 통합을 연결하여 AI 애플리케이션 개발을 간소화하며, 기술이 발전함에 따라 미래에 대비할 수 있는 결정을 내리는 데 도움을 줍니다. LangChain은 모듈화, 안정성, 그리고 검증된 성능을 제공하는 `langchain-core`를 기반으로 하여, 다양한 모델 제공자가 요구하는 인터페이스를 구현할 수 있도록 설계되었습니다. 이를 통해 LangChain 생태계 내에서 쉽게 사용될 수 있습니다.'

In [24]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

In [28]:
openai_model:str = 'gpt-4o'
temperature:float = 0.0

In [25]:
class Goal(BaseModel):
    description: str = Field(..., description="목표설명")

    @property
    def text(self) -> str:
        return f"{self.description}"

In [30]:
class PassiveGoalCreator:
    def __init__(self,llm:ChatOpenAI):
        self.llm = llm
    
    def run(self, query:str) -> Goal:
        prompt = ChatPromptTemplate.from_template(
            "사용자 입력을 분석하여 명확하고 실행 가능한 목표를 생성해 주세요.\n"
            "요건:\n"
            "1. 목표는 구체적이고 명확해야 하며, 실행 가능한 수준으로 상세화되어야 합니다.\n"
            "2. 당신이 실행할 수 있는 행동은 다음과 같은 행동뿐입니다.\n"
            "   - 인터넷을 이용하여 목표 달성을 위한 조사를 수행합니다.\n"
            "   - 사용자를 위한 보고서를 생성합니다.\n"
            "3. 절대 2.에 명시된 행동 외의 다른 행동을 취해서는 안 됩니다.\n"
            "사용자 입력: {query}"
        )
        chain = prompt | self.llm.with_structured_output(Goal)

        return chain.invoke({'query':query})

In [31]:
task = '삼성전자 주식에 대해 평가해줘.'

In [32]:
llm = ChatOpenAI(model = openai_model, temperature=temperature)
goal_creator = PassiveGoalCreator(llm=llm)
result:Goal = goal_creator.run(query=task)
print(f"{result.text}")

### 목표: 삼성전자 주식 평가 보고서 작성

#### 1. 목표 설명
삼성전자 주식에 대한 포괄적이고 명확한 평가 보고서를 작성하여 사용자에게 제공한다. 이 보고서는 삼성전자의 현재 주식 상태, 시장 동향, 경쟁사 비교, 그리고 미래 전망을 포함해야 한다.

#### 2. 목표 달성을 위한 세부 단계

1. **삼성전자 주식의 현재 상태 조사**
   - 삼성전자의 최근 주가 변동 및 거래량 분석
   - 최근 분기 실적 및 재무제표 검토
   - 주요 뉴스 및 발표 사항 확인

2. **시장 동향 및 경쟁사 분석**
   - 반도체 및 전자 산업의 최근 트렌드 조사
   - 주요 경쟁사(예: 애플, 인텔 등)와의 비교 분석
   - 글로벌 경제 상황이 삼성전자에 미치는 영향 평가

3. **미래 전망 및 전문가 의견 수집**
   - 주식 전문가 및 애널리스트의 삼성전자 주식에 대한 의견 수집
   - 삼성전자의 미래 계획 및 전략 분석
   - 기술 혁신 및 신제품 출시 계획 조사

4. **보고서 작성 및 제공**
   - 수집한 정보를 바탕으로 명확하고 체계적인 보고서 작성
   - 보고서에는 삼성전자 주식의 강점, 약점, 기회, 위협(SWOT 분석) 포함
   - 사용자가 이해하기 쉽게 요약 및 결론 제시

#### 3. 목표 달성의 기대 결과
사용자는 삼성전자 주식에 대한 명확하고 실행 가능한 정보를 얻어 투자 결정을 내리는 데 도움을 받을 수 있다. 이 보고서는 사용자가 삼성전자 주식의 현재 상태와 미래 가능성을 이해하는 데 기여할 것이다.


### 연습 한번 더 하기

In [39]:
class OptimizedGoal(BaseModel):
    description: str = Field(..., description="목표설명")
    metrics:str = Field(..., description = '목표의 달성도를 측정하는 방법')

    @property
    def text(self) -> str:
        return f"{self.description}(\n\n측정 기준:{self.metrics})"

In [40]:
class PromptOptimizer:
    def __init__(self,llm:ChatOpenAI):
        self.llm = llm
    
    def run(self, query:str) -> OptimizedGoal:
        prompt = ChatPromptTemplate.from_template(
            "당신은 목표 설정 전문가입니다. 아래의 목표를 SMART 원칙(Specific: 구체적, Measurable: 측정 가능, Achievable: 달성 가능, Relevant: 관련성이 높은, Time-bound: 기한이 있는)에 기반하여 최적화해 주세요.\n\n"
            "원래 목표:\n"
            "{query}\n\n"
            "지시 사항:\n"
            "1. 원래 목표를 분석하고, 부족한 요소나 개선점을 파악해 주세요.\n"
            "2. 당신이 실행할 수 있는 행동은 다음과 같습니다.\n"
            "   - 인터넷을 이용하여 목표 달성을 위한 조사를 수행한다.\n"
            "   - 사용자를 위한 보고서를 생성한다.\n"
            "3. SMART 원칙의 각 요소를 고려하면서 목표를 구체적이고 상세하게 기술해 주세요.\n"
            "   - 절대 추상적인 표현을 포함해서는 안 됩니다.\n"
            "   - 반드시 모든 단어가 실행 가능하고 구체적인지 확인해 주세요.\n"
            "4. 목표의 달성도를 측정하는 방법을 구체적이고 상세하게 기술해 주세요.\n"
            "5. 원래 목표에서 기한이 지정되지 않은 경우에는 기한을 고려할 필요가 없습니다.\n"
            "6. 주의: 절대로 2번 이외의 행동을 취해서는 안 됩니다."
        )
        chain = prompt | self.llm.with_structured_output(OptimizedGoal)

        return chain.invoke({'query':query})

In [41]:
llm = ChatOpenAI(model= openai_model, temperature=temperature)
passive_goal_creator = PassiveGoalCreator(llm=llm)
goal: Goal = passive_goal_creator.run(query=task)
print(f"{goal.text}")
prompt_optimizer = PromptOptimizer(llm=llm)
optimized_goal: OptimizedGoal = prompt_optimizer.run(query=goal.text)
print(f"{optimized_goal.text}")

삼성전자 주식에 대한 평가를 수행하여 투자 결정을 내릴 수 있도록 구체적이고 실행 가능한 보고서를 작성한다.(

측정 기준:1. 삼성전자 주식의 최근 1년간 주가 변동 추세를 분석하여 그래프로 시각화한다.\n2. 삼성전자의 재무제표를 분석하여 주요 재무 지표(예: 매출 성장률, 순이익률, 부채비율 등)를 도출한다.\n3. 삼성전자의 산업 내 경쟁사와의 비교 분석을 통해 시장 내 위치를 평가한다.\n4. 삼성전자의 향후 6개월간 주가 예측을 위한 전문가 의견 및 시장 전망을 조사하여 요약한다.\n5. 위의 분석 결과를 바탕으로 삼성전자 주식의 투자 적합성을 평가하고, 투자 권장 여부를 명확히 제시한다.)
삼성전자 주식에 대한 평가를 수행하여 투자 결정을 내릴 수 있도록 구체적이고 실행 가능한 보고서를 작성한다.(

측정 기준:1. **주가 변동 추세 분석 및 시각화**: 삼성전자 주식의 최근 1년간 주가 데이터를 수집하여 월별 변동 추세를 그래프로 시각화한다. 이를 통해 주가의 상승 및 하락 패턴을 명확히 파악한다.

2. **재무제표 분석**: 삼성전자의 최근 3년간 재무제표를 분석하여 매출 성장률, 순이익률, 부채비율 등 주요 재무 지표를 도출한다. 각 지표의 연도별 변화를 표로 정리하여 시각적으로 비교한다.

3. **경쟁사 비교 분석**: 삼성전자의 주요 경쟁사(예: LG전자, SK하이닉스 등)와의 재무 지표 및 시장 점유율을 비교 분석한다. 이를 통해 삼성전자의 시장 내 위치를 평가하고, 경쟁 우위 및 약점을 도출한다.

4. **전문가 의견 및 시장 전망 조사**: 삼성전자의 향후 6개월간 주가 예측을 위해 3명의 금융 전문가의 의견을 조사하고, 시장 전망 보고서를 분석하여 요약한다. 각 전문가의 예측을 비교하여 공통된 전망을 도출한다.

5. **투자 적합성 평가 및 권장 여부 제시**: 위의 분석 결과를 바탕으로 삼성전자 주식의 투자 적합성을 평가한다. 투자 권장 여부를 명확히 제시하고, 그 근거를 구체적으로 설명한다.

**기한**: 모든 분석 